# Perch → EfficientNet distillation · Colab

Select a GPU runtime. Locally build `uv run python -m scripts.colab.wheelhouse build`.
Upload to `/content/drive/Shareddrives/BirdCLEF2026/`:
- `birdclef-colab.zip`: current package and prepared labels/folds;
- `data/birdclef-audio.zip`: `train_audio/` and `train_soundscapes/` at the ZIP root;
- `data/cache/perch-cache.zip`: the existing Perch cache with `index.json`.

Audio and cache are unpacked on the VM; runs and checkpoints are saved on Drive.
Start with `SMOKE_RUN=True`, then set it to `False` for the real experiment.


In [ ]:
from pathlib import Path
import shutil
from zipfile import ZipFile

from google.colab import drive

drive.mount("/content/drive")
ROOT = Path("/content/drive/Shareddrives/BirdCLEF2026")
WORK = Path("/content/birdclef")
AUDIO_ZIP = ROOT / "data/birdclef-audio.zip"


def unpack(source, destination):
    local = Path("/content") / source.name
    shutil.copyfile(source, local)
    with ZipFile(local) as archive:
        archive.extractall(destination)
    local.unlink()


unpack(ROOT / "birdclef-colab.zip", WORK / "bundle")
unpack(AUDIO_ZIP, WORK / "competition")
unpack(ROOT / "data/cache/perch-cache.zip", WORK / "cache")
assert (WORK / "competition/train_audio").is_dir()
assert (WORK / "competition/train_soundscapes").is_dir()

bundle = WORK / "bundle"
wheel, = bundle.glob("birdclef*.whl")
%pip install --find-links "{bundle}" -r "{bundle}/requirements.txt" -q
%pip install --force-reinstall --no-deps "{wheel}" -q


In [ ]:
import json
import os

import torch

from birdclef.config import DataConfig, TrainConfig, get_config
from birdclef.models import AudioExperimentConfig, AudioModelConfig
from birdclef.models.distillation.config import DistillationConfig
from birdclef.models.encoders.download import download_perch
from birdclef.runtime import apply_environment_defaults

apply_environment_defaults()
torch.set_num_threads(4)
assert torch.cuda.is_available(), "Select a GPU runtime before training."

SMOKE_RUN = False
WANDB_OFFLINE = SMOKE_RUN
NAME = "distillation/colab-d1"
PREPARED_DIR = bundle / "data_prep"
TEACHER_PATH = WORK / "cache/models/perch_v2_backbone.onnx"
download_perch(TEACHER_PATH)
data = DataConfig(
    competition_dir=WORK / "competition",
    cache_dir=WORK / "cache",
    output_dir=ROOT / "outputs",
)
if not WANDB_OFFLINE:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    # If your W&B account uses a custom server, also set WANDB_BASE_URL here.

loader = {
    "batch_size": 4 if SMOKE_RUN else 64,
    "val_batch_size": 16, "predict_batch_size": 16,
    "num_workers": 0, "pin_memory": False,
    "seed": 42, "soundscape_probability": 0.1,
}
runtime = {"accelerator": "gpu", "devices": 1}


def save_config(config, name):
    path = config.data.output_dir / "configs" / f"{config.run_name(name)}.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(config.safe_dump(), indent=2), encoding="utf-8")
    return path


## Stage 1: features

One centered 5-second crop per focal clip; exact soundscape windows. Reads the existing cache only. Validation recordings are held out of training. Best checkpoint minimizes validation cosine loss.


In [ ]:
feature_config = get_config(
    DistillationConfig,
    teacher_path=TEACHER_PATH,
    data=data, fold=0, smoke=SMOKE_RUN,
    model=AudioModelConfig(encoder_lr=3e-4),
    loader=loader, runtime=runtime,
    tracking={"offline": WANDB_OFFLINE},
    train=TrainConfig(
        max_epochs=1 if SMOKE_RUN else 5,
        early_stopping_patience=2,
        log_every_n_steps=1 if SMOKE_RUN else 10,
    ),
)
FEATURE_NAME = f"{NAME}/features"
FEATURE_DIR = feature_config.data.output_dir / feature_config.run_name(FEATURE_NAME)
feature_path = save_config(feature_config, FEATURE_NAME)


In [ ]:
%run -m birdclef.models.distillation.fit -- --config "{feature_path}" --name "{FEATURE_NAME}" --prepared-dir "{PREPARED_DIR}"


## Stage 2: supervised full fine-tuning

Load only `encoder.pt`; discard the projection and initialize a fresh linear head for 234 classes. Both encoder and head use LR `3e-4` (D1). Normal random focal crops resume. Select the best checkpoint by validation ROC-AUC.


In [ ]:
classifier_config = get_config(
    AudioExperimentConfig,
    data=data, fold=feature_config.fold, smoke=SMOKE_RUN,
    model=AudioModelConfig(
        encoder_weights=FEATURE_DIR / "encoder.pt",
        encoder_lr=3e-4, head_lr=3e-4,
        unfreeze_epoch=0, unfreeze_last_n=None,
    ),
    loader=loader, runtime=runtime,
    tracking={"offline": WANDB_OFFLINE},
    train=TrainConfig(
        max_epochs=1 if SMOKE_RUN else 10,
        checkpoint_metric="val/auc", checkpoint_mode="max",
        early_stopping_patience=3,
        log_every_n_steps=1 if SMOKE_RUN else 10,
    ),
)
CLASSIFIER_NAME = f"{NAME}/full-ft"
RUN_DIR = classifier_config.data.output_dir / classifier_config.run_name(CLASSIFIER_NAME)
classifier_path = save_config(classifier_config, CLASSIFIER_NAME)


In [ ]:
%run -m birdclef.models.fit -- --config "{classifier_path}" --name "{CLASSIFIER_NAME}" --prepared-dir "{PREPARED_DIR}"


In [ ]:
import pandas as pd
from IPython.display import display

display(pd.read_json(RUN_DIR / "validation-metrics.json", typ="series"))
print(f"Classifier for export: {RUN_DIR}")


## Export

The ZIP is saved on Drive. Upload it to the models Dataset and update the wheelhouse
Dataset with the current package. The submission notebook remains unchanged.
Export only Stage 2; Perch and its cache are not required for inference.
Select the student with `MODEL_WEIGHTS = {"full-ft": 1.0}`.


In [ ]:
%run -m birdclef.submission.export -- --run "{RUN_DIR}" --all --output "{ROOT}/perch-distilled-efficientnet.zip"
